In [ ]:
%matplotlib inline

# NOTE: %reset wipes every variable in the current kernel session, and normally
# prompts "Proceed (y/[n])?" interactively when run in a live notebook - kept
# here to ensure a clean run, but will hang if this notebook is ever run
# non-interactively (e.g. via jupyter nbconvert --execute).
%reset

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

import hashlib
import h5py
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import pickle
import plotly.graph_objects as go
import warnings

from collections import Counter, defaultdict
from joblib import Parallel, delayed
from pathlib import Path
from scipy.io import loadmat
from scipy.sparse import csr_matrix, eye as sparse_eye, kron
from scipy.sparse import diags as sparse_diags
from scipy.sparse.csgraph import connected_components as sp_connected_components
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests

import sys
sys.path.append(str(projectRoot / 'functions'))  # so the import below resolves

from contrastTfceHybridFunctionsPermLabelOnly import (
    build_contrast_trials_hybrid,
    process_one_infant_chrom_hybrid,
)

projectRoot = Path().resolve().parent

# ---- USER SETTINGS ----
# Root of the drive/mount holding the adjacency matrices (the same `driveName`
# used throughout the imageReconstruction MATLAB repo). Edit for your machine —
# pathlib.Path handles the rest cross-platform, no OS-specific branching needed.
dataDriveRoot = projectRoot / 'auxilliary'


In [ ]:
#User defined params
preProcPipeline = 'Standard' #'Standard, '015LPF, '025LPF'...

trialsIncluded = 'Full' # 'Hab', 'Nov, or 'Full'
firstTrial = 1
if trialsIncluded == 'Hab':
    lastTrial = 15
elif trialsIncluded == 'Nov':
    lastTrial = 20
else:
    lastTrial = 25

loadOnly = 1  # set to 1 to load existing results if available, 0 to always recompute

# Column name for the per-parcel summary t-statistic. Spatiotemporal pipeline
# stores the peak |t| across the test window as 't peak'; older spatial-only
# results used a scalar 't'. Use whichever is present.
T_COL = 't peak'

numPermutations = 1000

ttestSignificance = 0.05
cptSignificance = 0.01
# multiple comparison correction methods - e.g. 'b' = Bonferroni
ttestCorrectMethod = 'fdr_bh'
withinChanCorrectMethod = 'h' #'h' Holm
numBasis = 21
firstSamp = -40
lastSamp = 189
parcels = 'ALL' #'ROI' for ROI channels base on previous analysis; 'ALL' for all array channels
useLongRun = False #see autocorrelation test and ACF figures below!
lrCovLag = 0 #set to 0 to calculate h based on dataset size
useWildBinSeg = True

lineStyles = [ '-', '--', '-.', ':', 'solid', 'dashed', 'dashdot', 'dotted'] # for plotting

#for indexing later on
firstFunc = firstTrial-1
lastFunc = lastTrial-1


In [ ]:
parcelFilenameMetadata = projectRoot / 'dataRaw' / f'indigoSurfParcelMetadata{preProcPipeline}.csv'
parcelFilenameTimeseries = projectRoot / 'dataRaw' / f'indigoSurfParcelTrialsVlen{preProcPipeline}.h5'

outputFilenameHabCombos = projectRoot / 'data' / f'indigoIndividualSurfParcelHabCombosDf{preProcPipeline}{trialsIncluded}.csv'

# Consolidated output directory + backward-compatible string form (kept as a
# plain string, matching the many 'DATA + "filename.csv"' string-concatenation
# calls used throughout this notebook, so those don't all need rewriting).
# Previously this was hardcoded and redefined three separate times in the
# original notebook (once here, once before the masking section, once before
# the group-level tests) - consolidated to one definition, here, up front.
tablesDir = projectRoot / 'data'
tablesDir.mkdir(parents=True, exist_ok=True)
DATA = str(tablesDir) + os.sep


In [ ]:
# Load metadata CSV
metadataDf = pd.read_csv(parcelFilenameMetadata)

# load time series data
with h5py.File(parcelFilenameTimeseries, 'r') as h5f:
    tsDataset = h5f['timeSeries'][()].copy()

In [ ]:
# ---- Load pre-built adjacency matrices ----
ADJACENCY_PATH = dataDriveRoot / 'mri' / 'adjacencyMatrices' / 'surfAdjacenciesAllAges.pkl'

with open(ADJACENCY_PATH, 'rb') as f:
    saved_adjacencies = pickle.load(f)

# Convert to (adjacency, parcel_order) tuples per age — what the pipeline expects
adjacencies = {
    age: (entry['adjacency'], entry['parcel_order'])
    for age, entry in saved_adjacencies.items()
}

print(f"Loaded adjacencies for ages: {sorted(adjacencies.keys())}")
for age, (adj, order) in adjacencies.items():
    print(f"  Age {age}: {adj.shape[0]} parcels, {adj.sum() // 2} edges")

T-TESTS

In [ ]:
# NOTE ON NAMING: the functions below (compute_tstat_spatiotemporal,
# build_spatiotemporal_adjacency, compute_spatiotemporal_tfce_fast,
# maxstat_tfce_permutation) are this notebook's OWN implementation for
# ACTIVATION testing (one-sample t vs baseline, circular-shift null).
# The imported contrastTfceHybridFunctionsPermLabelOnly module has its own,
# separately-defined functions with some of the SAME names, used for CONTRAST
# testing (Welch t between two trial groups, exhaustive label-permutation
# null) - a genuinely different statistical procedure, not a duplicate. Only
# build_contrast_trials_hybrid and process_one_infant_chrom_hybrid are
# actually imported from that module (see the imports cell), so there's no
# runtime name collision - but it's worth knowing these are two separate
# implementations before assuming one is dead code or trying to merge them.

# =============================================================================
# CORE FUNCTIONS
# =============================================================================

def compute_tstat_spatiotemporal(trial_data, baseline_idx, test_idx):
    """One-sample t per timepoint vs each trial's own pre-stim baseline mean."""
    baseline_mean = np.nanmean(trial_data[:, baseline_idx], axis=1, keepdims=True)
    centred = trial_data[:, test_idx] - baseline_mean

    n_valid = np.sum(~np.isnan(centred), axis=0)
    mean = np.nanmean(centred, axis=0)
    sd = np.nanstd(centred, axis=0, ddof=1)

    with np.errstate(invalid='ignore', divide='ignore'):
        t = mean / (sd / np.sqrt(n_valid))
    t[(n_valid < 2) | (sd == 0)] = np.nan
    return t


def circular_shift_trials(data_3d, rng):
    """Independent random circular shift per trial along time axis."""
    n_parcels, n_trials, n_samples = data_3d.shape
    shifts = rng.integers(0, n_samples, size=n_trials)
    shifted = np.empty_like(data_3d)
    for t in range(n_trials):
        shifted[:, t, :] = np.roll(data_3d[:, t, :], shifts[t], axis=-1)
    return shifted


# =============================================================================
# SPATIOTEMPORAL ADJACENCY (precomputed once per task)
# =============================================================================

def build_spatiotemporal_adjacency(spatial_adj, n_time, include_diagonals=True):
    """
    Build a sparse (n_parcels * n_time) x (n_parcels * n_time) adjacency matrix.
    Node ordering: row-major over (parcel, timepoint), i.e. node_idx = p*n_time + t.

    Connectivity:
      - spatial:  (p, t) ↔ (q, t)   if spatial_adj[p, q]
      - temporal: (p, t) ↔ (p, t±1)
      - diagonal: (p, t) ↔ (q, t±1) if spatial_adj[p, q]   (only if include_diagonals)

    include_diagonals=True matches the ept_TFCE (Mensen) convention, where the
    channel-neighbour scan is nested inside the t-1, t, t+1 loop, so a node
    connects to spatial neighbours at adjacent timepoints too. False gives the
    face-connected scheme (spatial OR temporal step, not both at once).
    """
    n_parcels = spatial_adj.shape[0]
    sp = csr_matrix(spatial_adj.astype(bool))

    I_time = sparse_eye(n_time, dtype=bool, format='csr')
    # T_off: tridiagonal with 1s on ±1 off-diagonals (temporal neighbours)
    T_off = sparse_diags([np.ones(n_time - 1, dtype=bool), np.ones(n_time - 1, dtype=bool)],
                         offsets=[-1, 1], shape=(n_time, n_time), format='csr', dtype=bool)

    # spatial at same t:  spatial_adj ⊗ I_time
    spatial_part = kron(sp, I_time, format='csr')
    # temporal at same parcel:  I_parcels ⊗ T_off
    temporal_part = kron(sparse_eye(n_parcels, dtype=bool, format='csr'), T_off, format='csr')

    full = spatial_part + temporal_part
    if include_diagonals:
        # spatial neighbour AND temporal step simultaneously: spatial_adj ⊗ T_off
        diagonal_part = kron(sp, T_off, format='csr')
        full = full + diagonal_part

    return full.astype(bool).tocsr()


# =============================================================================
# SPATIOTEMPORAL TFCE (vectorised connected components)
# =============================================================================

def compute_spatiotemporal_tfce_fast(t_map_2d, st_adj, e=1.0, h=2.0, n_steps=50):
    n_parcels, n_time = t_map_2d.shape
    n_nodes = n_parcels * n_time
    tfce_flat = np.zeros(n_nodes)
    t_flat = t_map_2d.ravel()
    valid = ~np.isnan(t_flat)
    if not valid.any():
        return tfce_flat.reshape(n_parcels, n_time)

    fmax = np.nanmax(t_flat[valid])          # positive tail only
    if fmax <= 0:
        return tfce_flat.reshape(n_parcels, n_time)
    delta = fmax / n_steps
    thresholds = (np.arange(n_steps) + 0.5) * delta

    for thresh in thresholds:
        supra = (t_flat >= thresh) & valid
        if not supra.any():
            continue
        supra_idx = np.where(supra)[0]
        sub_adj = st_adj[supra_idx][:, supra_idx]
        n_comp, labels = sp_connected_components(sub_adj, directed=False)
        sizes = np.bincount(labels, minlength=n_comp)
        increments = (sizes ** e) * (thresh ** h) * delta
        tfce_flat[supra_idx] += increments[labels]

    return tfce_flat.reshape(n_parcels, n_time)


# =============================================================================
# MAX-STAT SPATIOTEMPORAL TFCE PERMUTATION
# =============================================================================

def maxstat_tfce_permutation(trial_data_by_parcel, adjacency, parcel_order,
                              baseline_idx, test_idx, chrom_sign=+1,
                              n_perms=10000, e=1.0, h=2.0, n_steps=50,
                              include_diagonals=True,
                              fdr_q=0.05, rng=None, verbose=False):
    """
    Returns
    -------
    obs_t        : dict {parcel: ndarray (n_test,)}
    obs_tfce     : dict {parcel: ndarray (n_test,)}
    perm_p       : dict {parcel: ndarray (n_test,)}  per-POINT permutation p (with +1 correction)
    parcel_p_min : dict {parcel: min per-point p across window}
    threshold    : float, 95th pct of max|TFCE| null  (FWER)
    max_null     : ndarray (n_perms,)
    fwer_p       : dict {parcel: ndarray (n_test,)}   global-max FWER p (with +1 correction)
    """
    if rng is None:
        rng = np.random.default_rng(42)

    parcels = [p for p in parcel_order if p in trial_data_by_parcel]
    if not parcels:
        return {}, {}, {}, {}, np.nan, np.array([]), {}

    parcel_to_pos = {p: i for i, p in enumerate(parcel_order)}
    keep_idx = [parcel_to_pos[p] for p in parcels]
    adj_sub = adjacency[np.ix_(keep_idx, keep_idx)]

    data_3d = np.stack([trial_data_by_parcel[p] for p in parcels], axis=0)
    n_parcels, n_trials, n_samples = data_3d.shape

    n_test = len(range(*test_idx.indices(n_samples)))
    st_adj = build_spatiotemporal_adjacency(adj_sub, n_test,
                                            include_diagonals=include_diagonals)

    # ---- Observed statistics ----
    obs_t_2d = chrom_sign * np.array([
        compute_tstat_spatiotemporal(data_3d[i], baseline_idx, test_idx)
        for i in range(n_parcels)
    ])
    obs_tfce_2d = compute_spatiotemporal_tfce_fast(obs_t_2d, st_adj, e=e, h=h, n_steps=n_steps)
    obs_abs = np.abs(obs_tfce_2d)
    obs_finite = ~np.isnan(obs_tfce_2d)

    obs_t    = {p: obs_t_2d[i]    for i, p in enumerate(parcels)}
    obs_tfce = {p: obs_tfce_2d[i] for i, p in enumerate(parcels)}

    # ---- Permutation loop: build BOTH the global-max null and a per-point count ----
    max_null = np.zeros(n_perms)
    # per-point count of permutations whose |TFCE| >= observed |TFCE| at that point
    ge_count = np.zeros_like(obs_tfce_2d, dtype=np.int64)

    for i in range(n_perms):
        shifted = circular_shift_trials(data_3d, rng)
        perm_t_2d = chrom_sign * np.array([
            compute_tstat_spatiotemporal(shifted[j], baseline_idx, test_idx)
            for j in range(n_parcels)
        ])
        perm_tfce_2d = compute_spatiotemporal_tfce_fast(perm_t_2d, st_adj, e=e, h=h, n_steps=n_steps)
        perm_abs = np.abs(perm_tfce_2d)

        valid = ~np.isnan(perm_abs)
        max_null[i] = np.max(perm_abs[valid]) if valid.any() else 0.0

        # per-point tally (only where both observed and perm are finite)
        cmp = obs_finite & valid & (perm_abs >= obs_abs)
        ge_count[cmp] += 1

        if verbose and (i + 1) % 1000 == 0:
            print(f"    Permutation {i+1}/{n_perms}")

    threshold = np.percentile(max_null, 95)

    # ---- Per-POINT permutation p-values, with +1 correction (#9) ----
    point_p_2d = np.full_like(obs_tfce_2d, np.nan, dtype=float)
    point_p_2d[obs_finite] = (1 + ge_count[obs_finite]) / (n_perms + 1)

    # ---- Global-max FWER p-values, with +1 correction (#9) ----
    fwer_p_2d = np.full_like(obs_tfce_2d, np.nan, dtype=float)
    if obs_finite.any():
        av = obs_abs[obs_finite]
        # (1 + #{max_null >= |obs|}) / (n_perms + 1)
        ge_max = (max_null[None, :] >= av[:, None]).sum(axis=1)
        fwer_p_2d[obs_finite] = (1 + ge_max) / (n_perms + 1)

    perm_p   = {p: point_p_2d[i] for i, p in enumerate(parcels)}
    fwer_p   = {p: fwer_p_2d[i]  for i, p in enumerate(parcels)}
    parcel_p_min = {
        p: (float(np.nanmin(point_p_2d[i])) if np.any(~np.isnan(point_p_2d[i])) else np.nan)
        for i, p in enumerate(parcels)
    }

    return obs_t, obs_tfce, perm_p, parcel_p_min, threshold, max_null, fwer_p


# =============================================================================
# SANITY CHECKS
# =============================================================================

def check_null_distribution(max_null):
    if len(max_null) == 0:
        return {}
    return {
        'mean': float(np.mean(max_null)),
        'sd': float(np.std(max_null)),
        'median': float(np.median(max_null)),
        'p95': float(np.percentile(max_null, 95)),
        'p99': float(np.percentile(max_null, 99)),
        'min': float(np.min(max_null)),
        'max': float(np.max(max_null)),
    }


def effect_size_check_spatiotemporal(trial_data_by_parcel, sig_mask_by_parcel,
                                      stim_onset, baseline_idx, test_idx):
    """
    Cohen's d (paired) computed over the trial-mean of significant timepoints
    within each parcel: trial mean over significant samples minus per-trial
    baseline mean, divided by SD across trials.
    """
    effect_sizes = {}
    for p, mask in sig_mask_by_parcel.items():
        if p not in trial_data_by_parcel or not mask.any():
            continue
        trials = trial_data_by_parcel[p]
        baselines = np.nanmean(trials[:, baseline_idx], axis=1)
        test_window = trials[:, test_idx]                 # (n_trials, n_test_samples)
        sig_means = np.nanmean(test_window[:, mask], axis=1)  # (n_trials,)
        diffs = sig_means - baselines
        diffs = diffs[~np.isnan(diffs)]
        if len(diffs) < 2 or diffs.std(ddof=1) == 0:
            effect_sizes[p] = np.nan
        else:
            effect_sizes[p] = float(diffs.mean() / diffs.std(ddof=1))
    return effect_sizes


# =============================================================================
# PER-INFANT PIPELINE
# =============================================================================

def run_infant_analysis(trial_data_by_parcel, adjacency, parcel_order,
                        stim_onset=40, test_start_offset=30, test_end_offset=None,
                        chrom_sign=+1,
                        n_perms=10000, n_steps=50, include_diagonals=True,
                        fdr_q=0.05, verbose=False, rng_seed=42):
    """
    Per-infant spatiotemporal TFCE pipeline (circular-shift null).
    Produces BOTH FWER (global max-stat) and FDR (per-point BH) significance.
    """
    rng = np.random.default_rng(rng_seed)
    parcels = list(trial_data_by_parcel.keys())
    if not parcels:
        return None

    shapes = [trial_data_by_parcel[p].shape for p in parcels]
    modal_n_trials = Counter(s[0] for s in shapes).most_common(1)[0][0]
    modal_n_samples = Counter(s[1] for s in shapes).most_common(1)[0][0]

    consistent_parcels = {
        p: trial_data_by_parcel[p]
        for p in parcels
        if trial_data_by_parcel[p].shape == (modal_n_trials, modal_n_samples)
    }
    n_dropped = len(parcels) - len(consistent_parcels)
    if n_dropped > 0 and verbose:
        print(f"  Dropped {n_dropped} parcels with inconsistent shape")
    if not consistent_parcels:
        return None

    clean_parcels = {
        p: trials for p, trials in consistent_parcels.items()
        if not np.isnan(trials).any()
    }
    if not clean_parcels:
        return None
    if verbose:
        print(f"  Parcels after cleaning: {len(clean_parcels)}")

    baseline_end = stim_onset
    test_start = stim_onset + test_start_offset
    test_end = modal_n_samples if test_end_offset is None else stim_onset + test_end_offset

    if baseline_end <= 0 or test_start >= test_end:
        warnings.warn("Window indices invalid. Skipping infant.")
        return None
    if test_end > modal_n_samples:
        warnings.warn(f"Test window beyond series (end={test_end}, len={modal_n_samples}). Skipping.")
        return None

    baseline_idx = slice(0, baseline_end)
    test_idx = slice(test_start, test_end)

    obs_t, obs_tfce, perm_p, parcel_p_min, threshold, max_null, fwer_p = maxstat_tfce_permutation(
        clean_parcels, adjacency, parcel_order,
        baseline_idx, test_idx, chrom_sign=chrom_sign,
        n_perms=n_perms, e=1.0, h=2.0, n_steps=n_steps,
        include_diagonals=include_diagonals, fdr_q=fdr_q,
        rng=rng, verbose=verbose
    )

    null_stats = check_null_distribution(max_null)
    if verbose:
        print(f"  Null distribution: {null_stats}")

    parcels_kept = list(obs_t.keys())

    # ---- FWER significance: per-point fwer_p < 0.05 ----
    fwer_mask = {p: (~np.isnan(fwer_p[p])) & (fwer_p[p] < 0.05) for p in parcels_kept}

    # ---- FDR significance: per-PARCEL (collapse time, BH across parcels) ----
    # Each parcel is summarised by its single most-significant per-point p
    # (min over its timepoints). BH is then applied across PARCELS only, so the
    # family size is ~n_parcels (e.g. 200-280), not n_parcels*n_time (~26000).
    # This removes the per-pixel BH "tipping point" that flipped whole infants
    # between all-significant and none. A parcel survives if its best point
    # clears the parcel-level BH critical value; within a surviving parcel, the
    # flagged timepoints are those whose own per-point p is at/below that value.
    if parcels_kept:
        parcel_min_p = np.array([
            np.nanmin(perm_p[p]) if np.any(~np.isnan(perm_p[p])) else np.nan
            for p in parcels_kept
        ])
        crit_p = benjamini_hochberg_critical_p(parcel_min_p, q=fdr_q)
        fdr_mask = {
            p: ((~np.isnan(perm_p[p])) & (perm_p[p] <= crit_p))
            for p in parcels_kept
        }
    else:
        fdr_mask = {}

    # effect sizes computed over the FDR-significant timepoints (more sensitive set)
    effect_sizes = effect_size_check_spatiotemporal(
        clean_parcels, fdr_mask, stim_onset, baseline_idx, test_idx
    )

    sig_parcels_fwer = [p for p, m in fwer_mask.items() if m.any()]
    sig_parcels_fdr  = [p for p, m in fdr_mask.items()  if m.any()]

    return {
        'observed_t': obs_t,
        'observed_tfce': obs_tfce,
        'permutation_p': perm_p,         # per-point p
        'fwer_p': fwer_p,                # global-max p
        'parcel_p_min': parcel_p_min,
        'fwer_mask': fwer_mask,
        'fdr_mask': fdr_mask,
        'fwer_threshold': threshold,
        'fdr_q': fdr_q,
        'tfce_params': {'e': 1.0, 'h': 2.0, 'n_steps': n_steps,
                        'include_diagonals': include_diagonals},
        'null_method': 'circular_shift_per_trial_spatiotemporal',
        'max_null': max_null,
        'null_distribution_stats': null_stats,
        'effect_sizes': effect_sizes,
        'test_window_samples': (test_start, test_end),
        'baseline_window_samples': (0, baseline_end),
        'n_parcels_input': len(parcels),
        'n_parcels_tested': len(clean_parcels),
        'n_parcels_dropped_shape': n_dropped,
        'n_significant_parcels_fwer': len(sig_parcels_fwer),
        'n_significant_parcels_fdr':  len(sig_parcels_fdr),
        'n_significant_pixels_fwer': int(sum(m.sum() for m in fwer_mask.values())),
        'n_significant_pixels_fdr':  int(sum(m.sum() for m in fdr_mask.values())),
        'frac_parcels_sig_fdr': (
            len([p for p, m in fdr_mask.items() if m.any()]) / max(len(clean_parcels), 1)
        ),
        'fdr_critical_p': crit_p if parcels_kept else np.nan,
    }


# =============================================================================
# HELPERS FOR THE MAIN LOOP
# =============================================================================

def _deterministic_seed(*parts) -> int:
    key = "|".join(str(p) for p in parts).encode("utf-8")
    return int(hashlib.md5(key).hexdigest()[:8], 16)


def build_cleaned(infant_chrom_df, tsDataset, stim_onset):
    """Unchanged."""
    trial_data_by_parcel = defaultdict(list)
    for parcel, parcel_group in infant_chrom_df.groupby('Parcel'):
        for idx in parcel_group['tsIDx']:
            ts = tsDataset[idx] * 1e6
            if np.isnan(ts).all():
                continue
            trial_data_by_parcel[parcel].append(ts)

    parcel_shapes = []
    for parcel, trial_list in trial_data_by_parcel.items():
        if len(trial_list) >= 3:
            lengths = [len(t) for t in trial_list]
            parcel_shapes.append((len(trial_list), min(lengths)))

    if not parcel_shapes:
        return {}

    modal_n_trials = Counter(s[0] for s in parcel_shapes).most_common(1)[0][0]
    modal_n_samples = Counter(s[1] for s in parcel_shapes).most_common(1)[0][0]

    cleaned = {}
    for parcel, trial_list in trial_data_by_parcel.items():
        if len(trial_list) < 3:
            continue
        truncated = [t[:modal_n_samples] for t in trial_list if len(t) >= modal_n_samples]
        if len(truncated) < modal_n_trials:
            continue
        cleaned[parcel] = np.array(truncated[:modal_n_trials])

    return cleaned

def benjamini_hochberg_mask(pvals_flat, q=0.05):
    """
    Benjamini-Hochberg FDR. Returns a boolean mask (same shape as input) that is
    True for rejected (significant) hypotheses. NaNs are treated as non-significant
    and excluded from the m used in the BH denominator.
    """
    mask = np.zeros(pvals_flat.shape, dtype=bool)
    finite = ~np.isnan(pvals_flat)
    p = pvals_flat[finite]
    m = p.size
    if m == 0:
        return mask

    order = np.argsort(p)
    p_sorted = p[order]
    thresh = (np.arange(1, m + 1) / m) * q
    below = p_sorted <= thresh
    if not below.any():
        return mask  # nothing survives
    k = np.max(np.where(below)[0])           # largest index satisfying BH
    crit_p = p_sorted[k]
    rej_finite = p <= crit_p

    full_rej = np.zeros(m, dtype=bool)
    full_rej[rej_finite] = True
    mask[finite] = full_rej
    return mask

def benjamini_hochberg_critical_p(pvals, q=0.05):
    """
    Return the BH critical p-value: the largest p-value that is rejected at FDR=q.
    Returns 0.0 if nothing is rejected. NaNs are excluded from the family size.
    """
    p = pvals[~np.isnan(pvals)]
    m = p.size
    if m == 0:
        return 0.0
    p_sorted = np.sort(p)
    below = p_sorted <= (np.arange(1, m + 1) / m) * q
    if not below.any():
        return 0.0
    return float(p_sorted[np.max(np.where(below)[0])])


# =============================================================================
# ACTIVATION — process_one_infant_chrom 
# =============================================================================
#   - compute_spatiotemporal_tfce_fast: positive tail only
#   - maxstat_tfce_permutation: accepts chrom_sign, signs the t-map
#   - run_infant_analysis: accepts chrom_sign, passes it through

def process_one_infant_chrom(age, infant, chrom, cleaned, adjacency, parcel_order, cfg):
    """Worker: runs the spatiotemporal pipeline for one infant×chrom (Fam1 activation)."""
    seed = _deterministic_seed(age, infant, chrom)

    CHROM_SIGN = {1: +1, 2: -1}            # HbO: increase; HbR: decrease
    chrom_sign = CHROM_SIGN.get(chrom, +1)

    result = run_infant_analysis(
        cleaned,
        adjacency=adjacency,
        parcel_order=parcel_order,
        stim_onset=cfg['stim_onset'],
        test_start_offset=cfg['test_start_offset'],
        test_end_offset=cfg.get('test_end_offset'),
        chrom_sign=chrom_sign,
        n_perms=cfg['n_perms'],
        n_steps=cfg.get('n_steps', 50),
        include_diagonals=cfg.get('include_diagonals', True),
        fdr_q=cfg.get('fdr_q', 0.05),
        verbose=False,
        rng_seed=seed,
    )
    if result is None:
        return [], None

    test_start, test_end = result['test_window_samples']
    n_test_samples = test_end - test_start

    results_rows = []
    for parcel in result['observed_t'].keys():
        trials_arr = cleaned[parcel]
        n_trials_used = trials_arr.shape[0]
        baselines_raw = np.nanmean(trials_arr[:, 0:cfg['stim_onset']], axis=1)

        t_vec     = result['observed_t'][parcel]
        tfce_vec  = result['observed_tfce'][parcel]
        pp_vec    = result['permutation_p'][parcel]
        fwerp_vec = result['fwer_p'][parcel]
        fwer_mask = result['fwer_mask'][parcel]
        fdr_mask  = result['fdr_mask'][parcel]

        if np.any(~np.isnan(t_vec)):
            best_idx = int(np.nanargmax(np.abs(t_vec)))
            t_peak = float(t_vec[best_idx])
            tfce_peak = float(tfce_vec[best_idx])
            pp_at_peak = float(pp_vec[best_idx])
            fwerp_at_peak = float(fwerp_vec[best_idx])
        else:
            best_idx = -1
            t_peak = tfce_peak = pp_at_peak = fwerp_at_peak = np.nan

        n_sig_fwer = int(fwer_mask.sum())
        n_sig_fdr  = int(fdr_mask.sum())
        any_fwer = bool(n_sig_fwer > 0)
        any_fdr  = bool(n_sig_fdr > 0)

        # per-timepoint significant sample indices (WITHIN the test window),
        # stored as a comma-separated string for the temporal-overlap step.
        sig_idx_fdr = np.where(fdr_mask)[0]
        sig_idx_fdr_str = ','.join(map(str, sig_idx_fdr.tolist()))

        report_mask = fdr_mask if any_fdr else fwer_mask
        if report_mask.any():
            sig_means = np.nanmean(trials_arr[:, test_start:test_end][:, report_mask], axis=1)
            mean_sig = float(np.nanmean(sig_means))
            sd_sig   = float(np.nanstd(sig_means))
        else:
            mean_sig = sd_sig = np.nan

        results_rows.append({
            'Age': age,
            'Parcel': parcel,
            'Chrom': chrom,
            'Infant': infant,
            'Mean Baseline': float(np.mean(baselines_raw)),
            'SD Baseline': float(np.std(baselines_raw)),
            'Mean Sig Window': mean_sig,
            'SD Sig Window':  sd_sig,
            't peak': t_peak,                 # signed & oriented: positive = hypothesised
            'TFCE peak': tfce_peak,
            'p at peak (point)': pp_at_peak,
            'p at peak (FWER)':  fwerp_at_peak,
            'Peak sample idx (within test)': best_idx,
            'N sig samples FWER': n_sig_fwer,
            'N sig samples FDR':  n_sig_fdr,
            'Sig sample idx FDR': sig_idx_fdr_str,   # <-- NEW (temporal overlap)
            'N test samples': n_test_samples,
            'Test start sample': test_start,         # <-- NEW (grid alignment check)
            'n': n_trials_used,
            'Significant FWER': any_fwer,
            'Significant FDR':  any_fdr,
            'Significant': any_fdr,
        })

    diag_row = {
        'Age': age,
        'Infant': infant,
        'Chrom': chrom,
        'Null method': result['null_method'],
        'Chrom sign': chrom_sign,
        'Null mean': result['null_distribution_stats'].get('mean'),
        'Null sd': result['null_distribution_stats'].get('sd'),
        'Null p95 threshold': result['null_distribution_stats'].get('p95'),
        'FDR q': result['fdr_q'],
        'Test window start sample': test_start,
        'Test window end sample': test_end,
        'N parcels tested': result['n_parcels_tested'],
        'N significant parcels FWER': result['n_significant_parcels_fwer'],
        'N significant parcels FDR':  result['n_significant_parcels_fdr'],
        'N significant pixels FWER':  result['n_significant_pixels_fwer'],
        'N significant pixels FDR':   result['n_significant_pixels_fdr'],
        'Frac parcels sig FDR': result.get('frac_parcels_sig_fdr'),
        'FDR critical p': result.get('fdr_critical_p'),
    }
    return results_rows, diag_row


Activation

In [ ]:
# =============================================================================
# MAIN LOOP
# =============================================================================

# Configuration
stimOnset = 40
TEST_START_OFFSET = 30          # 3 sec post-stim at fs=10Hz
TEST_END_OFFSET   = None        # None -> to end of trial
N_PERMUTATIONS = numPermutations
SIGNIFICANCE_ALPHA = ttestSignificance
N_JOBS = -1

cfg = {
    'stim_onset': stimOnset,
    'test_start_offset': TEST_START_OFFSET,
    'test_end_offset':   TEST_END_OFFSET,
    'n_perms': N_PERMUTATIONS,
    'alpha': SIGNIFICANCE_ALPHA,
    'n_steps': 50,               # midpoint integration steps
    'include_diagonals': True,   # ept_TFCE-style connectivity 
    'fdr_q': 0.05,               # Benjamini-Hochberg target FDR
}

results_path = tablesDir / "surf_parcel_TFCE_circshift_results_INDIV_spatiotemporal_FDR.csv"
diagnostics_path = tablesDir / "surf_parcel_TFCE_circshift_diagnostics_INDIV_spatiotemporal_FDR.csv"

if loadOnly == 1 and os.path.exists(results_path):
    print(f"Loading existing results from {results_path}")
    resultsDf = pd.read_csv(results_path)
    if os.path.exists(diagnostics_path):
        diagnosticsDf = pd.read_csv(diagnostics_path)
        print(f"Loaded existing diagnostics from {diagnostics_path}")
    else:
        diagnosticsDf = pd.DataFrame()
        print(f"  No diagnostics file found at {diagnostics_path}")
else:
    if loadOnly == 1:
        print(f"loadOnly=1 but no file found at {results_path} — running computation")
    
    # ---- Build task list ----
    tasks = []
    for age in sorted(metadataDf['Age'].unique()):
        if age not in adjacencies:
            print(f"  No adjacency available for age={age} — skipping all infants at this age")
            continue
        
        adjacency, parcel_order = adjacencies[age]
        
        ageDf = metadataDf[(metadataDf['Age'] == age) & (metadataDf['Trial'] <= 8)]
        for (infant, chrom), infant_chrom_df in ageDf.groupby(['Infant', 'Chrom']):
            cleaned = build_cleaned(infant_chrom_df, tsDataset, stimOnset)
            if not cleaned:
                print(f"  No usable parcels for age={age}, infant={infant}, chrom={chrom}")
                continue
            tasks.append((age, infant, chrom, cleaned, adjacency, parcel_order))

    print(f"Dispatching {len(tasks)} infant×chrom tasks across workers...")

    outputs = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
        delayed(process_one_infant_chrom)(age, infant, chrom, cleaned, adjacency, parcel_order, cfg)
        for (age, infant, chrom, cleaned, adjacency, parcel_order) in tasks
    )

    results = []
    diagnostics = []
    for rows, diag in outputs:
        results.extend(rows)
        if diag is not None:
            diagnostics.append(diag)

    resultsDf = pd.DataFrame(results)
    diagnosticsDf = pd.DataFrame(diagnostics)

    resultsDf.to_csv(results_path, index=False)
    diagnosticsDf.to_csv(diagnostics_path, index=False)

print("\n=== Summary ===")
print(f"Total parcel results: {len(resultsDf)}")
print(f"Significant (FWER): {resultsDf['Significant FWER'].sum()}")
print(f"Significant (FDR):  {resultsDf['Significant FDR'].sum()}")

In [ ]:
act = resultsDf[resultsDf['Significant'] == True]
print(act.groupby('Chrom')['t peak'].apply(lambda s: (s > 0).mean()))

Habituation

In [ ]:
# =============================================================================
# MAIN LOOP — HABITUATION TFCE (NULL: LABEL-PERM)
# =============================================================================
#
# Contrast: hab1 (trials 1-5) vs hab3 (trials 11-15)
# Direction: positive t = hab1 > hab3 (habituation)
# Min trials per group: 3
# Post-hoc filter: parcels significant in resultsDf (activation)
# =============================================================================

# Configuration
stimOnset, peakOnset, peakOffset = 40, 70, 110
N_PERMUTATIONS = numPermutations
SIGNIFICANCE_ALPHA = ttestSignificance
N_JOBS = -1
MIN_PER_GROUP = 3

GROUP_A_TRIALS = (1, 2, 3, 4, 5)        # hab1
GROUP_B_TRIALS = (11, 12, 13, 14, 15)   # hab3
DIRECTION      = 'a_minus_b'            # positive t = hab1 > hab3
GROUP_A_LABEL  = 'Hab1'
GROUP_B_LABEL  = 'Hab3'
SEED_TAG       = 'hab_hybrid'
MAX_TRIAL_NEEDED = max(max(GROUP_A_TRIALS), max(GROUP_B_TRIALS))

cfg_hab = {
    'stim_onset': stimOnset,
    'test_start_offset': 30,   # 3 sec post-stim at fs=10 Hz
    'test_end_offset': None,   # None → end of trial
    'n_perms': N_PERMUTATIONS,
    'alpha': SIGNIFICANCE_ALPHA,
}

hab_results_path = tablesDir / "surf_parcel_TFCE_habituation_hybrid_results_INDIV_spatiotemporal_perm_label.csv"
hab_diagnostics_path = tablesDir / "surf_parcel_TFCE_habituation_hybrid_diagnostics_INDIV_spatiotemporal_perm_label.csv"
hab_intersection_path = tablesDir / "surf_parcel_TFCE_habituation_hybrid_AND_activation_INDIV_spatiotemporal_perm_label.csv"

if loadOnly == 1 and os.path.exists(hab_results_path):
    print(f"Loading existing habituation (hybrid) results from {hab_results_path}")
    habDf = pd.read_csv(hab_results_path)
    if os.path.exists(hab_diagnostics_path):
        habDiagnosticsDf = pd.read_csv(hab_diagnostics_path)
        print(f"Loaded existing habituation (hybrid) diagnostics from {hab_diagnostics_path}")
    else:
        habDiagnosticsDf = pd.DataFrame()
else:
    if loadOnly == 1:
        print(f"loadOnly=1 but no file found at {hab_results_path} — running computation")
    
    tasks = []
    for age in sorted(metadataDf['Age'].unique()):
        if age not in adjacencies:
            print(f"  No adjacency available for age={age} — skipping")
            continue
        
        adjacency, parcel_order = adjacencies[age]
        ageDf = metadataDf[(metadataDf['Age'] == age) &
                           (metadataDf['Trial'] <= MAX_TRIAL_NEEDED)]
        
        for (infant, chrom), infant_chrom_df in ageDf.groupby(['Infant', 'Chrom']):
            cleaned, n_a, n_b = build_contrast_trials_hybrid(
                infant_chrom_df, tsDataset,
                group_a_trials=GROUP_A_TRIALS,
                group_b_trials=GROUP_B_TRIALS,
                min_per_group=MIN_PER_GROUP,
            )
            if not cleaned:
                print(f"  No usable parcels for age={age}, infant={infant}, chrom={chrom}")
                continue
            tasks.append((age, infant, chrom, cleaned, n_a, n_b,
                          adjacency, parcel_order))
    
    print(f"Dispatching {len(tasks)} infant×chrom habituation tasks across workers...")
    
    outputs = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
        delayed(process_one_infant_chrom_hybrid)(
            age, infant, chrom, cleaned, n_a, n_b,
            adjacency, parcel_order, cfg_hab,
            seed_tag=SEED_TAG,
            group_a_label=GROUP_A_LABEL,
            group_b_label=GROUP_B_LABEL,
            direction=DIRECTION,
        )
        for (age, infant, chrom, cleaned, n_a, n_b, adjacency, parcel_order) in tasks
    )
    
    hab_results = []
    hab_diagnostics = []
    for rows, diag in outputs:
        hab_results.extend(rows)
        if diag is not None:
            hab_diagnostics.append(diag)
    
    habDf = pd.DataFrame(hab_results)
    habDiagnosticsDf = pd.DataFrame(hab_diagnostics)
    
    habDf.to_csv(hab_results_path, index=False)
    habDiagnosticsDf.to_csv(hab_diagnostics_path, index=False)
    
    print("\n=== Habituation (hybrid null) summary ===")
    print(f"Total parcel results: {len(habDf)}")
    print(f"Total significant: {habDf['Significant'].sum()}")




Novelty


In [ ]:
# =============================================================================
# MAIN LOOP — NOVELTY TFCE (NULL: LABEL-PERM)
# =============================================================================
#
# Contrast: hab3 (trials 11-15) vs nov (trials 16-20)
# Direction: positive t = nov > hab3 (release)
# Min trials per group: 3
# Post-hoc filter: parcels significant in BOTH resultsDf (activation)
#                  AND habDf (habituation)
# =============================================================================

# Configuration
stimOnset, peakOnset, peakOffset = 40, 70, 110
N_PERMUTATIONS = numPermutations
SIGNIFICANCE_ALPHA = ttestSignificance
N_JOBS = -1
MIN_PER_GROUP = 3

GROUP_A_TRIALS = (11, 12, 13, 14, 15)   # hab3
GROUP_B_TRIALS = (16, 17, 18, 19, 20)   # nov
DIRECTION      = 'b_minus_a'            # positive t = nov > hab3
GROUP_A_LABEL  = 'Hab3'
GROUP_B_LABEL  = 'Nov'
SEED_TAG       = 'nov_hybrid'
MAX_TRIAL_NEEDED = max(max(GROUP_A_TRIALS), max(GROUP_B_TRIALS))

cfg_nov = {
    'stim_onset': stimOnset,
    'test_start_offset': 30,   # 3 sec post-stim at fs=10 Hz
    'test_end_offset': None,   # None → end of trial
    'n_perms': N_PERMUTATIONS,
    'alpha': SIGNIFICANCE_ALPHA,
}

nov_results_path = tablesDir / "surf_parcel_TFCE_novelty_hybrid_results_INDIV_spatiotemporal_perm_label.csv"
nov_diagnostics_path = tablesDir / "surf_parcel_TFCE_novelty_hybrid_diagnostics_INDIV_spatiotemporal_perm_label.csv"
nov_intersection_path = tablesDir / "surf_parcel_TFCE_novelty_hybrid_AND_activation_AND_habituation_INDIV_spatiotemporal_perm_label.csv"

if loadOnly == 1 and os.path.exists(nov_results_path):
    print(f"Loading existing novelty (hybrid) results from {nov_results_path}")
    novDf = pd.read_csv(nov_results_path)
    if os.path.exists(nov_diagnostics_path):
        novDiagnosticsDf = pd.read_csv(nov_diagnostics_path)
        print(f"Loaded existing novelty (hybrid) diagnostics from {nov_diagnostics_path}")
    else:
        novDiagnosticsDf = pd.DataFrame()
else:
    if loadOnly == 1:
        print(f"loadOnly=1 but no file found at {nov_results_path} — running computation")
    
    tasks = []
    for age in sorted(metadataDf['Age'].unique()):
        if age not in adjacencies:
            print(f"  No adjacency available for age={age} — skipping")
            continue
        
        adjacency, parcel_order = adjacencies[age]
        ageDf = metadataDf[(metadataDf['Age'] == age) &
                           (metadataDf['Trial'] <= MAX_TRIAL_NEEDED)]
        
        for (infant, chrom), infant_chrom_df in ageDf.groupby(['Infant', 'Chrom']):
            cleaned, n_a, n_b = build_contrast_trials_hybrid(
                infant_chrom_df, tsDataset,
                group_a_trials=GROUP_A_TRIALS,
                group_b_trials=GROUP_B_TRIALS,
                min_per_group=MIN_PER_GROUP,
            )
            if not cleaned:
                print(f"  No usable parcels for age={age}, infant={infant}, chrom={chrom}")
                continue
            tasks.append((age, infant, chrom, cleaned, n_a, n_b,
                          adjacency, parcel_order))
    
    print(f"Dispatching {len(tasks)} infant×chrom novelty tasks across workers...")
    
    outputs = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
        delayed(process_one_infant_chrom_hybrid)(
            age, infant, chrom, cleaned, n_a, n_b,
            adjacency, parcel_order, cfg_nov,
            seed_tag=SEED_TAG,
            group_a_label=GROUP_A_LABEL,
            group_b_label=GROUP_B_LABEL,
            direction=DIRECTION,
        )
        for (age, infant, chrom, cleaned, n_a, n_b, adjacency, parcel_order) in tasks
    )
    
    nov_results = []
    nov_diagnostics = []
    for rows, diag in outputs:
        nov_results.extend(rows)
        if diag is not None:
            nov_diagnostics.append(diag)
    
    novDf = pd.DataFrame(nov_results)
    novDiagnosticsDf = pd.DataFrame(nov_diagnostics)
    
    novDf.to_csv(nov_results_path, index=False)
    novDiagnosticsDf.to_csv(nov_diagnostics_path, index=False)
    
    print("\n=== Novelty (hybrid null) summary ===")
    print(f"Total parcel results: {len(novDf)}")
    print(f"Total significant: {novDf['Significant'].sum()}")


Extended Habituation


In [ ]:
# =============================================================================
# MAIN LOOP — DELAYED HABITUATION TFCE (NULL: LABEL-PERM)
# =============================================================================
#
# Contrast: hab1 (trials 1-5) vs nov (trials 16-20)
# Direction: positive t = hab1 > nov (sustained reduction)
# Min trials per group: 3
# Post-hoc filter: parcels significant in resultsDf (activation)
# =============================================================================

# Configuration
stimOnset, peakOnset, peakOffset = 40, 70, 110
N_PERMUTATIONS = numPermutations
SIGNIFICANCE_ALPHA = ttestSignificance
N_JOBS = -1
MIN_PER_GROUP = 3

GROUP_A_TRIALS = (1, 2, 3, 4, 5)        # hab1
GROUP_B_TRIALS = (16, 17, 18, 19, 20)   # nov
DIRECTION      = 'a_minus_b'            # positive t = hab1 > nov
GROUP_A_LABEL  = 'Hab1'
GROUP_B_LABEL  = 'Nov'
SEED_TAG       = 'delayhab_hybrid'
MAX_TRIAL_NEEDED = max(max(GROUP_A_TRIALS), max(GROUP_B_TRIALS))

cfg_delayhab = {
    'stim_onset': stimOnset,
    'test_start_offset': 30,   # 3 sec post-stim at fs=10 Hz
    'test_end_offset': None,   # None → end of trial
    'n_perms': N_PERMUTATIONS,
    'alpha': SIGNIFICANCE_ALPHA,
}

delayhab_results_path = tablesDir / "surf_parcel_TFCE_delayedHab_hybrid_results_INDIV_spatiotemporal_perm_label.csv"
delayhab_diagnostics_path = tablesDir / "surf_parcel_TFCE_delayedHab_hybrid_diagnostics_INDIV_spatiotemporal_perm_label.csv"
delayhab_intersection_path = tablesDir / "surf_parcel_TFCE_delayedHab_hybrid_AND_activation_INDIV_spatiotemporal_perm_label.csv"

if loadOnly == 1 and os.path.exists(delayhab_results_path):
    print(f"Loading existing delayed habituation (hybrid) results from {delayhab_results_path}")
    delayHabDf = pd.read_csv(delayhab_results_path)
    if os.path.exists(delayhab_diagnostics_path):
        delayHabDiagnosticsDf = pd.read_csv(delayhab_diagnostics_path)
    else:
        delayHabDiagnosticsDf = pd.DataFrame()
else:
    if loadOnly == 1:
        print(f"loadOnly=1 but no file found at {delayhab_results_path} — running computation")
    
    tasks = []
    for age in sorted(metadataDf['Age'].unique()):
        if age not in adjacencies:
            print(f"  No adjacency available for age={age} — skipping")
            continue
        
        adjacency, parcel_order = adjacencies[age]
        ageDf = metadataDf[(metadataDf['Age'] == age) &
                           (metadataDf['Trial'] <= MAX_TRIAL_NEEDED)]
        
        for (infant, chrom), infant_chrom_df in ageDf.groupby(['Infant', 'Chrom']):
            cleaned, n_a, n_b = build_contrast_trials_hybrid(
                infant_chrom_df, tsDataset,
                group_a_trials=GROUP_A_TRIALS,
                group_b_trials=GROUP_B_TRIALS,
                min_per_group=MIN_PER_GROUP,
            )
            if not cleaned:
                print(f"  No usable parcels for age={age}, infant={infant}, chrom={chrom}")
                continue
            tasks.append((age, infant, chrom, cleaned, n_a, n_b,
                          adjacency, parcel_order))
    
    print(f"Dispatching {len(tasks)} infant×chrom delayed-habituation tasks across workers...")
    
    outputs = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
        delayed(process_one_infant_chrom_hybrid)(
            age, infant, chrom, cleaned, n_a, n_b,
            adjacency, parcel_order, cfg_delayhab,
            seed_tag=SEED_TAG,
            group_a_label=GROUP_A_LABEL,
            group_b_label=GROUP_B_LABEL,
            direction=DIRECTION,
        )
        for (age, infant, chrom, cleaned, n_a, n_b, adjacency, parcel_order) in tasks
    )
    
    delayhab_results = []
    delayhab_diagnostics = []
    for rows, diag in outputs:
        delayhab_results.extend(rows)
        if diag is not None:
            delayhab_diagnostics.append(diag)
    
    delayHabDf = pd.DataFrame(delayhab_results)
    delayHabDiagnosticsDf = pd.DataFrame(delayhab_diagnostics)
    
    delayHabDf.to_csv(delayhab_results_path, index=False)
    delayHabDiagnosticsDf.to_csv(delayhab_diagnostics_path, index=False)
    
    print("\n=== Delayed habituation (hybrid null) summary ===")
    print(f"Total parcel results: {len(delayHabDf)}")
    print(f"Total significant: {delayHabDf['Significant'].sum()}")


'Long-range' Habituation

In [ ]:
# =============================================================================
# MAIN LOOP — LONG-RANGE HABITUATION TFCE (NULL: LABEL-PERM)
# =============================================================================
#
# Contrast: hab1 (trials 1-5) vs post (trials 21-25)
# Direction: positive t = hab1 > post (persistent suppression)
# Min trials per group: 3
# Post-hoc filter: parcels significant in resultsDf (activation)
# =============================================================================

# Configuration
stimOnset, peakOnset, peakOffset = 40, 70, 110
N_PERMUTATIONS = numPermutations
SIGNIFICANCE_ALPHA = ttestSignificance
N_JOBS = -1
MIN_PER_GROUP = 3

GROUP_A_TRIALS = (1, 2, 3, 4, 5)        # hab1
GROUP_B_TRIALS = (21, 22, 23, 24, 25)   # post
DIRECTION      = 'a_minus_b'            # positive t = hab1 > post
GROUP_A_LABEL  = 'Hab1'
GROUP_B_LABEL  = 'Post'
SEED_TAG       = 'lrhab_hybrid'
MAX_TRIAL_NEEDED = max(max(GROUP_A_TRIALS), max(GROUP_B_TRIALS))

cfg_lrhab = {
    'stim_onset': stimOnset,
    'test_start_offset': 30,   # 3 sec post-stim at fs=10 Hz
    'test_end_offset': None,   # None → end of trial
    'n_perms': N_PERMUTATIONS,
    'alpha': SIGNIFICANCE_ALPHA,
}

lrhab_results_path = tablesDir / "surf_parcel_TFCE_lrHab_hybrid_results_INDIV_spatiotemporal_perm_label.csv"
lrhab_diagnostics_path = tablesDir / "surf_parcel_TFCE_lrHab_hybrid_diagnostics_INDIV_spatiotemporal_perm_label.csv"
lrhab_intersection_path = tablesDir / "surf_parcel_TFCE_lrHab_hybrid_AND_activation_INDIV_spatiotemporal_perm_label.csv"

if loadOnly == 1 and os.path.exists(lrhab_results_path):
    print(f"Loading existing long-range habituation (hybrid) results from {lrhab_results_path}")
    lrHabDf = pd.read_csv(lrhab_results_path)
    if os.path.exists(lrhab_diagnostics_path):
        lrHabDiagnosticsDf = pd.read_csv(lrhab_diagnostics_path)
    else:
        lrHabDiagnosticsDf = pd.DataFrame()
else:
    if loadOnly == 1:
        print(f"loadOnly=1 but no file found at {lrhab_results_path} — running computation")
    
    tasks = []
    for age in sorted(metadataDf['Age'].unique()):
        if age not in adjacencies:
            print(f"  No adjacency available for age={age} — skipping")
            continue
        
        adjacency, parcel_order = adjacencies[age]
        ageDf = metadataDf[(metadataDf['Age'] == age) &
                           (metadataDf['Trial'] <= MAX_TRIAL_NEEDED)]
        
        for (infant, chrom), infant_chrom_df in ageDf.groupby(['Infant', 'Chrom']):
            cleaned, n_a, n_b = build_contrast_trials_hybrid(
                infant_chrom_df, tsDataset,
                group_a_trials=GROUP_A_TRIALS,
                group_b_trials=GROUP_B_TRIALS,
                min_per_group=MIN_PER_GROUP,
            )
            if not cleaned:
                print(f"  No usable parcels for age={age}, infant={infant}, chrom={chrom}")
                continue
            tasks.append((age, infant, chrom, cleaned, n_a, n_b,
                          adjacency, parcel_order))
    
    print(f"Dispatching {len(tasks)} infant×chrom long-range habituation tasks across workers...")
    
    if len(tasks) == 0:
        print("  No tasks — likely no data in trial range 21-25. "
              "Check metadataDf['Trial'].max().")
        lrHabDf = pd.DataFrame()
        lrHabDiagnosticsDf = pd.DataFrame()
    else:
        outputs = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
            delayed(process_one_infant_chrom_hybrid)(
                age, infant, chrom, cleaned, n_a, n_b,
                adjacency, parcel_order, cfg_lrhab,
                seed_tag=SEED_TAG,
                group_a_label=GROUP_A_LABEL,
                group_b_label=GROUP_B_LABEL,
                direction=DIRECTION,
            )
            for (age, infant, chrom, cleaned, n_a, n_b, adjacency, parcel_order) in tasks
        )
        
        lrhab_results = []
        lrhab_diagnostics = []
        for rows, diag in outputs:
            lrhab_results.extend(rows)
            if diag is not None:
                lrhab_diagnostics.append(diag)
        
        lrHabDf = pd.DataFrame(lrhab_results)
        lrHabDiagnosticsDf = pd.DataFrame(lrhab_diagnostics)
        
        lrHabDf.to_csv(lrhab_results_path, index=False)
        lrHabDiagnosticsDf.to_csv(lrhab_diagnostics_path, index=False)
        
        print("\n=== Long-range habituation (hybrid null) summary ===")
        print(f"Total parcel results: {len(lrHabDf)}")
        print(f"Total significant: {lrHabDf['Significant'].sum()}")


In [ ]:
# Fixed per-chromophore direction for "response magnitude" sign.
# A positive raw response (HbO) decreasing in magnitude => first - second > 0.
# A negative raw response (HbR) decreasing in magnitude => first - second < 0.
# So the SIGN that corresponds to the hypothesised contrast direction is:
CHROM_DIRECTION = {1: +1, 2: -1}   # HbO, HbR


def _parse_idx(s):
    """Parse a comma-separated sample-index string to a set of ints."""
    if pd.isna(s) or s == '':
        return set()
    return set(int(x) for x in str(s).split(',') if x != '')


def _direction_ok(t_peak, chrom, contrast_kind):
    """
    Does the RAW signed contrast t-peak match the hypothesised direction?

    contrast_kind:
      'habituation'  : first > second in magnitude  (Fam1>Fam3, Fam1>Nov, Fam1>Post)
                       hypothesised raw sign = CHROM_DIRECTION[chrom]
      'novelty'      : Novel > Fam3 in magnitude (response RECOVERS)
                       i.e. magnitude INCREASES from Fam3 to Novel.
                       With direction='b_minus_a' upstream (positive t = Nov>Fam3
                       in raw terms for HbO), hypothesised raw sign for a magnitude
                       increase is CHROM_DIRECTION[chrom].
    Note: because the contrast 'direction' (a_minus_b / b_minus_a) was already
    applied upstream so that "positive t = the named first group larger in RAW
    terms", the hypothesised sign here is simply CHROM_DIRECTION[chrom] for both
    kinds. Inverted-polarity parcels are not handled (matches Anna: fixed rule).
    """
    if np.isnan(t_peak):
        return False
    want = CHROM_DIRECTION.get(chrom, +1)
    return (t_peak * want) > 0


def _check_grid_alignment(contrast_df, activation_df):
    if 'Test start sample' in contrast_df and 'Test start sample' in activation_df:
        c = contrast_df['Test start sample'].dropna().unique()
        a = activation_df['Test start sample'].dropna().unique()
        if len(c) and len(a) and not (set(c) == set(a)):
            raise ValueError(
                f"Test-grid mismatch: contrast start samples {sorted(c)} vs "
                f"activation start samples {sorted(a)}. Sample-index overlap "
                f"would be meaningless. Align stim_onset/test_start_offset."
            )


def mask_contrast(contrast_csv, activation_csv, contrast_kind,
                  habituation_masked_csv=None,
                  out_csv=None):
    """
    Apply Anna-style masking to one contrast.

    Parameters
    ----------
    contrast_csv : per-infant contrast results (raw, from the contrast worker)
    activation_csv : Fam1 activation results (resultsDf)
    contrast_kind : 'habituation' or 'novelty'
        'habituation' covers Fam1>Fam3, Fam1>Nov, Fam1>Post (all gate on Fam1 +
        overlap + direction). 'novelty' (Novel>Fam3) additionally requires the
        parcel to be in the habituation-masked set.
    habituation_masked_csv : required iff contrast_kind == 'novelty'.
        The already-masked Habituation output (this function's own out_csv for
        the Fam1>Fam3 run).
    out_csv : where to write the masked contrast.

    Returns the masked DataFrame (rows that survive, with mask columns added).
    """
    con = pd.read_csv(contrast_csv)
    act = pd.read_csv(activation_csv)
    _check_grid_alignment(con, act)

    keys = ['Age', 'Parcel', 'Chrom', 'Infant']

    # --- (1) Fam1 activation significant: keep activation sig rows + their indices
    act_sig = act[act['Significant'] == True][keys + ['Sig sample idx FDR']].copy()
    act_sig = act_sig.rename(columns={'Sig sample idx FDR': 'Act sig idx'})

    merged = con.merge(act_sig, on=keys, how='inner')   # (1) drops non-activated parcels
    if merged.empty:
        masked = merged.copy()
        masked['temporal_overlap'] = []
        masked['direction_ok'] = []
        masked['Masked significant'] = []
        if out_csv:
            masked.to_csv(out_csv, index=False)
        return masked

    # --- (2) temporal overlap between contrast sig samples and Fam1 activation sig samples
    def _overlap(row):
        con_idx = _parse_idx(row.get('Sig sample idx FDR', ''))
        act_idx = _parse_idx(row.get('Act sig idx', ''))
        return bool(con_idx) and bool(act_idx) and len(con_idx & act_idx) > 0
    merged['temporal_overlap'] = merged.apply(_overlap, axis=1)

    # --- (3) direction matches hypothesis given fixed per-chrom rule
    merged['direction_ok'] = merged.apply(
        lambda r: _direction_ok(r['t peak'], r['Chrom'], contrast_kind), axis=1
    )

    # contrast must be significant (its own FDR), overlap, and correct direction
    merged['Masked significant'] = (
        merged['Significant'].astype(bool)
        & merged['temporal_overlap']
        & merged['direction_ok']
    )

    # --- (0) novelty prerequisite: parcel must have passed habituation masking
    if contrast_kind == 'novelty':
        if habituation_masked_csv is None:
            raise ValueError("novelty masking requires habituation_masked_csv")
        hab = pd.read_csv(habituation_masked_csv)
        hab_pass = hab[hab['Masked significant'] == True][keys].drop_duplicates()
        hab_pass['_hab_ok'] = True
        merged = merged.merge(hab_pass, on=keys, how='left')
        merged['_hab_ok'] = merged['_hab_ok'].fillna(False)
        merged['Masked significant'] = merged['Masked significant'] & merged['_hab_ok']
        merged = merged.drop(columns=['_hab_ok'])

    if out_csv:
        merged.to_csv(out_csv, index=False)

    n_in = len(con)
    n_act = len(merged)
    n_pass = int(merged['Masked significant'].sum())
    print(f"[{contrast_kind}] {contrast_csv.split('/')[-1]}")
    print(f"    rows in: {n_in} | after Fam1-activation join: {n_act} | "
          f"masked-significant: {n_pass}")

    return merged

In [ ]:
ACT  = DATA + "surf_parcel_TFCE_circshift_results_INDIV_spatiotemporal_FDR.csv"   # Fam1 activation

# 1) Habituation first (novelty depends on it)
mask_contrast(
    DATA + "surf_parcel_TFCE_habituation_hybrid_results_INDIV_spatiotemporal_perm_label.csv",
    ACT, 'habituation',
    out_csv=DATA + "surf_parcel_TFCE_habituation_MASKED_INDIV_spatiotemporal_perm_label.csv",
)

# 2) Delayed habituation (Fam1>Nov)
mask_contrast(
    DATA + "surf_parcel_TFCE_delayedHab_hybrid_results_INDIV_spatiotemporal_perm_label.csv",
    ACT, 'habituation',
    out_csv=DATA + "surf_parcel_TFCE_delayedHab_MASKED_INDIV_spatiotemporal_perm_label.csv",
)

# 3) Long-range habituation (Fam1>Post)
mask_contrast(
    DATA + "surf_parcel_TFCE_lrHab_hybrid_results_INDIV_spatiotemporal_perm_label.csv",
    ACT, 'habituation',
    out_csv=DATA + "surf_parcel_TFCE_lrHab_MASKED_INDIV_spatiotemporal_perm_label.csv",
)

# 4) Novelty (Novel>Fam3) — needs Fam1 activation AND habituation-masked parcels
mask_contrast(
    DATA + "surf_parcel_TFCE_novelty_hybrid_results_INDIV_spatiotemporal_perm_label.csv",
    ACT, 'novelty',
    habituation_masked_csv=DATA + "surf_parcel_TFCE_habituation_MASKED_INDIV_spatiotemporal_perm_label.csv",
    out_csv=DATA + "surf_parcel_TFCE_novelty_MASKED_INDIV_spatiotemporal_perm_label.csv",
)

# NOTE: Early habituation (Fam1 vs Fam2) and Mid habituation (Fam2 vs Fam3)
# masking calls removed - these were exploratory test cells (see notebook
# history) and are not part of the paper's analysis (which covers exactly the
# five tests above: activation, habituation, delayed habituation, long-range
# habituation, novelty).


In [ ]:
# =============================================================================
# GROUP-LEVEL PREPARATION: PRELIMINARY DESCRIPTIVE COUNTS (all 5 tests)
# =============================================================================
# For each test, per cell:
#   n_tested        = infants with a row in that test's CSV at that grouping
#                     ("the test actually ran" — includes the >=3-pair rule)
#   n_significant   = infants whose per-infant flag is True
#   pct_significant = 100 * n_significant / n_tested
#
# For contrasts, n_significant uses 'Masked significant' — so the drop from
# n_tested -> n_significant reflects the activation (and habituation, for
# novelty) prerequisite chain.
#
# Written at 3 granularities:
#   PRELIM_counts_parcel_age_chrom.csv  (finest)
#   PRELIM_counts_parcel_age.csv        (pooled across chrom)
#   PRELIM_counts_age_chrom.csv         (whole-brain summary)
# =============================================================================

ACT_CSV   = DATA + "surf_parcel_TFCE_circshift_results_INDIV_spatiotemporal_FDR.csv"
HAB_CSV   = DATA + "surf_parcel_TFCE_habituation_MASKED_INDIV_spatiotemporal_perm_label.csv"
DELAY_CSV = DATA + "surf_parcel_TFCE_delayedHab_MASKED_INDIV_spatiotemporal_perm_label.csv"
LR_CSV    = DATA + "surf_parcel_TFCE_lrHab_MASKED_INDIV_spatiotemporal_perm_label.csv"
NOV_CSV   = DATA + "surf_parcel_TFCE_novelty_MASKED_INDIV_spatiotemporal_perm_label.csv"

# (test_name, csv_path, per-infant significance column)
TESTS = [
    ('activation',  ACT_CSV,   'Significant'),
    ('habituation', HAB_CSV,   'Masked significant'),
    ('delayedHab',  DELAY_CSV, 'Masked significant'),
    ('lrHab',       LR_CSV,    'Masked significant'),
    ('novelty',     NOV_CSV,   'Masked significant'),
]


def _load(path):
    if not os.path.exists(path):
        print(f"  MISSING: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


def _count_rows(df, group_cols, sig_col):
    """Per group: n_tested, n_significant, pct_significant."""
    if df is None or df.empty:
        return pd.DataFrame(columns=group_cols + ['n_tested', 'n_significant', 'pct_significant'])

    tested = (df.groupby(group_cols)['Infant']
                .nunique()
                .reset_index(name='n_tested'))

    sig_df = df[df[sig_col] == True]
    if sig_df.empty:
        sig = pd.DataFrame({**{c: [] for c in group_cols}, 'n_significant': []})
    else:
        sig = (sig_df.groupby(group_cols)['Infant']
                     .nunique()
                     .reset_index(name='n_significant'))

    out = tested.merge(sig, on=group_cols, how='left').fillna({'n_significant': 0})
    out['n_significant'] = out['n_significant'].astype(int)
    out['pct_significant'] = np.where(
        out['n_tested'] > 0, 100 * out['n_significant'] / out['n_tested'], np.nan
    )
    return out


all_fine, all_parcel_age, all_age_chrom = [], [], []

for test_name, path, sig_col in TESTS:
    print(f"\n=== {test_name} ===")
    df = _load(path)
    if df.empty:
        continue
    if sig_col not in df.columns:
        print(f"  WARNING: sig column '{sig_col}' not in {path} -- skipping")
        continue

    fine       = _count_rows(df, ['Parcel', 'Age', 'Chrom'], sig_col)
    parcel_age = _count_rows(df, ['Parcel', 'Age'],          sig_col)
    age_chrom  = _count_rows(df, ['Age', 'Chrom'],           sig_col)
    for x in (fine, parcel_age, age_chrom):
        x.insert(0, 'test', test_name)

    all_fine.append(fine)
    all_parcel_age.append(parcel_age)
    all_age_chrom.append(age_chrom)

    # printed whole-brain summary
    print("  Whole-brain summary (n_tested / n_significant / pct):")
    print(age_chrom.to_string(index=False))

prelimFineDf      = pd.concat(all_fine,       ignore_index=True) if all_fine       else pd.DataFrame()
prelimParcelAgeDf = pd.concat(all_parcel_age, ignore_index=True) if all_parcel_age else pd.DataFrame()
prelimAgeChromDf  = pd.concat(all_age_chrom,  ignore_index=True) if all_age_chrom  else pd.DataFrame()

if not prelimFineDf.empty:
    prelimFineDf.to_csv(DATA + "PRELIM_counts_parcel_age_chrom.csv", index=False)
if not prelimParcelAgeDf.empty:
    prelimParcelAgeDf.to_csv(DATA + "PRELIM_counts_parcel_age.csv",  index=False)
if not prelimAgeChromDf.empty:
    prelimAgeChromDf.to_csv(DATA + "PRELIM_counts_age_chrom.csv",    index=False)

print("\nWritten preliminary count CSVs:")
print(f"  {DATA}PRELIM_counts_parcel_age_chrom.csv")
print(f"  {DATA}PRELIM_counts_parcel_age.csv")
print(f"  {DATA}PRELIM_counts_age_chrom.csv")


In [ ]:
# =============================================================================
# Per-cell attrition — data-quality (A) and post-activation (B) filtering
# =============================================================================
# For each (Age × Chrom × Parcel × Test):
#   A) data-quality attrition (unconditional on activation)
#      n_pool_A     = infants with any valid trial data at that parcel within the
#                     contrast's trial window
#      n_tested_A   = infants who cleared min-trials and appear in the results CSV
#      pct_excluded_A = 100 × (n_pool_A - n_tested_A) / n_pool_A
#
#   B) post-activation attrition (conditional on activation)
#      n_pool_B     = infants with activation-significant at that parcel
#      n_tested_B   = subset of B who also appear in the contrast results CSV
#      pct_excluded_B = 100 × (n_pool_B - n_tested_B) / n_pool_B
# =============================================================================

# Trial windows per contrast
CONTRAST_TRIAL_WINDOWS = {
    'activation':  set(range(1, 9)),
    'habituation': set(range(1, 6))  | set(range(11, 16)),
    'delayedHab':  set(range(1, 6))  | set(range(16, 21)),
    'lrHab':       set(range(1, 6))  | set(range(21, 26)),
    'novelty':     set(range(11, 16)) | set(range(16, 21)),
}

# Results CSV paths (already defined earlier as ACT_CSV etc.; using TESTS list for consistency)
RESULTS_PATHS = {name: path for name, path, _ in TESTS}

# Activation-significant set: infants who reached activation significance at (Age, Chrom, Parcel)
act_results = pd.read_csv(RESULTS_PATHS['activation'])
act_significant = act_results[act_results['Significant'] == True]
act_sig_pairs = (act_significant
                  .groupby(['Age', 'Chrom', 'Parcel'])['Infant']
                  .apply(set)
                  .reset_index(name='act_sig_infants'))


def _valid_ts_pool(trial_window):
    """Return DataFrame of infants with any non-NaN trial at each (Age, Chrom, Parcel)."""
    pool = metadataDf[metadataDf['Trial'].isin(trial_window)].copy()
    keep_mask = np.array([
        not np.isnan(tsDataset[idx]).all()
        for idx in pool['tsIDx']
    ])
    pool = pool[keep_mask]
    return (pool
            .groupby(['Age', 'Chrom', 'Parcel'])['Infant']
            .apply(set)
            .reset_index(name='pool_infants'))


# Compute per-cell attrition for each test
attrition_rows_A = []
attrition_rows_B = []

for test_name in CONTRAST_TRIAL_WINDOWS:
    trial_window = CONTRAST_TRIAL_WINDOWS[test_name]
    csv_path = RESULTS_PATHS.get(test_name)
    if csv_path is None or not os.path.exists(csv_path):
        print(f"  Skipping {test_name}: results CSV not found")
        continue

    results_df = pd.read_csv(csv_path)

    # Tested at each (Age, Chrom, Parcel): infants with a row in the results CSV
    tested_pairs = (results_df
                     .groupby(['Age', 'Chrom', 'Parcel'])['Infant']
                     .apply(set)
                     .reset_index(name='tested_infants'))

    # (A) Data-quality attrition
    pool_A = _valid_ts_pool(trial_window)
    merged_A = pool_A.merge(tested_pairs,
                             on=['Age', 'Chrom', 'Parcel'],
                             how='left')
    merged_A['tested_infants'] = merged_A['tested_infants'].apply(
        lambda x: x if isinstance(x, set) else set())
    merged_A['n_pool_A']     = merged_A['pool_infants'].apply(len)
    merged_A['n_tested_A']   = merged_A['tested_infants'].apply(len)
    merged_A['n_excluded_A'] = merged_A['n_pool_A'] - merged_A['n_tested_A']
    merged_A['pct_excluded_A'] = 100.0 * merged_A['n_excluded_A'] / merged_A['n_pool_A']
    merged_A['test'] = test_name
    attrition_rows_A.append(
        merged_A[['test', 'Age', 'Chrom', 'Parcel',
                   'n_pool_A', 'n_tested_A', 'n_excluded_A', 'pct_excluded_A']]
    )

    # (B) Post-activation attrition — skip for activation itself
    if test_name == 'activation':
        continue

    merged_B = act_sig_pairs.merge(tested_pairs,
                                     on=['Age', 'Chrom', 'Parcel'],
                                     how='left')
    merged_B['tested_infants'] = merged_B['tested_infants'].apply(
        lambda x: x if isinstance(x, set) else set())
    # Restrict tested to those in the activation-significant pool
    merged_B['tested_and_act_sig'] = merged_B.apply(
        lambda r: r['tested_infants'] & r['act_sig_infants'], axis=1)
    merged_B['n_pool_B']     = merged_B['act_sig_infants'].apply(len)
    merged_B['n_tested_B']   = merged_B['tested_and_act_sig'].apply(len)
    merged_B['n_excluded_B'] = merged_B['n_pool_B'] - merged_B['n_tested_B']
    merged_B['pct_excluded_B'] = 100.0 * merged_B['n_excluded_B'] / merged_B['n_pool_B']
    merged_B['test'] = test_name
    attrition_rows_B.append(
        merged_B[['test', 'Age', 'Chrom', 'Parcel',
                   'n_pool_B', 'n_tested_B', 'n_excluded_B', 'pct_excluded_B']]
    )

attrition_A = pd.concat(attrition_rows_A, ignore_index=True)
attrition_B = (pd.concat(attrition_rows_B, ignore_index=True)
                 if attrition_rows_B else pd.DataFrame(
                     columns=['test', 'Age', 'Chrom', 'Parcel',
                              'n_pool_B', 'n_tested_B', 'n_excluded_B', 'pct_excluded_B']))

# Merge A and B into a single per-cell table
attritionDf = attrition_A.merge(
    attrition_B,
    on=['test', 'Age', 'Chrom', 'Parcel'],
    how='outer',
)

# Summary per (Age × Chrom × Test): median and IQR across parcels
print("=" * 100)
print("Per-cell attrition summary (medians across parcels within Age × Chrom)")
print("=" * 100)
for test_name in CONTRAST_TRIAL_WINDOWS:
    sub = attritionDf[attritionDf['test'] == test_name]
    if sub.empty:
        continue
    print(f"\n--- {test_name} ---")
    agg_cols = {
        'n_parcels':        ('Parcel', 'nunique'),
        'pool_A_median':    ('n_pool_A', 'median'),
        'tested_A_median':  ('n_tested_A', 'median'),
        'pct_excluded_A_median':  ('pct_excluded_A', 'median'),
    }
    if test_name != 'activation':
        agg_cols.update({
            'pool_B_median':          ('n_pool_B', 'median'),
            'tested_B_median':        ('n_tested_B', 'median'),
            'pct_excluded_B_median':  ('pct_excluded_B', 'median'),
        })
    grp = (sub.groupby(['Age', 'Chrom']).agg(**agg_cols).reset_index())
    print(grp.to_string(index=False))

# Save the per-cell table
attritionDf.to_csv(DATA + "PERCELL_attrition_A_and_B.csv", index=False)
print(f"\nWritten: {DATA}PERCELL_attrition_A_and_B.csv")

In [ ]:
# =============================================================================
# Combined table — whole-brain counts alongside attrition summary
# per (Age × Chrom × Test)
# =============================================================================
# Uses the previously-computed:
#   prelimAgeChromDf  : whole-brain summary (n_tested, n_significant, pct_significant)
#   attritionDf       : per-cell attrition (A and B)
# and outputs a single (Age × Chrom × Test) table for the paper / presentation.
# =============================================================================

# Aggregate attrition to (Age × Chrom × Test) using medians across parcels
attrition_summary = (
    attritionDf
    .groupby(['test', 'Age', 'Chrom'])
    .agg(pct_excluded_A_median=('pct_excluded_A', 'median'),
         pct_excluded_A_p25=('pct_excluded_A', lambda x: np.percentile(x.dropna(), 25) if x.notna().any() else np.nan),
         pct_excluded_A_p75=('pct_excluded_A', lambda x: np.percentile(x.dropna(), 75) if x.notna().any() else np.nan),
         pct_excluded_B_median=('pct_excluded_B', 'median'),
         pct_excluded_B_p25=('pct_excluded_B', lambda x: np.percentile(x.dropna(), 25) if x.notna().any() else np.nan),
         pct_excluded_B_p75=('pct_excluded_B', lambda x: np.percentile(x.dropna(), 75) if x.notna().any() else np.nan))
    .reset_index()
)

# Combine with the whole-brain summary
combinedDf = prelimAgeChromDf.merge(
    attrition_summary,
    on=['test', 'Age', 'Chrom'],
    how='left',
)

# Round percentages for readability
for col in ['pct_significant',
            'pct_excluded_A_median', 'pct_excluded_A_p25', 'pct_excluded_A_p75',
            'pct_excluded_B_median', 'pct_excluded_B_p25', 'pct_excluded_B_p75']:
    if col in combinedDf.columns:
        combinedDf[col] = combinedDf[col].round(1)

# Present in clean column order
col_order = ['test', 'Age', 'Chrom',
             'n_tested', 'n_significant', 'pct_significant',
             'pct_excluded_A_median', 'pct_excluded_A_p25', 'pct_excluded_A_p75',
             'pct_excluded_B_median', 'pct_excluded_B_p25', 'pct_excluded_B_p75']
combinedDf = combinedDf[[c for c in col_order if c in combinedDf.columns]]

print("=" * 130)
print("Combined table: whole-brain counts + per-cell attrition medians (IQR)")
print("=" * 130)
for test_name in TESTS:
    name = test_name[0]
    sub = combinedDf[combinedDf['test'] == name]
    if sub.empty:
        continue
    print(f"\n--- {name} ---")
    print(sub.to_string(index=False))

combinedDf.to_csv(DATA + "COMBINED_whole_brain_plus_attrition.csv", index=False)
print(f"\nWritten: {DATA}COMBINED_whole_brain_plus_attrition.csv")

In [ ]:
# =============================================================================
# Per-parcel attrition — how much data yield is lost at each parcel due to
# minimum-trial filtering, and (for contrasts) how much is lost between the
# activation-significant pool and the contrast results.
# =============================================================================

CONTRAST_TRIAL_WINDOWS = {
    'activation':  set(range(1, 9)),
    'habituation': set(range(1, 6))  | set(range(11, 16)),
    'delayedHab':  set(range(1, 6))  | set(range(16, 21)),
    'lrHab':       set(range(1, 6))  | set(range(21, 26)),
    'novelty':     set(range(11, 16)) | set(range(16, 21)),
}
RESULTS_PATHS = {name: path for name, path, _ in TESTS}

# --- Activation-significant sets per (Age, Chrom, Parcel) ---
act_results = pd.read_csv(RESULTS_PATHS['activation'])
act_sig = act_results[act_results['Significant'] == True]
act_sig_by_cell = (act_sig
                    .groupby(['Age', 'Chrom', 'Parcel'])['Infant']
                    .apply(set)
                    .reset_index(name='act_sig_infants'))


# --- Helper: raw pool of infants with any non-NaN trial at each parcel ---
def _raw_pool_by_cell(trial_window):
    pool = metadataDf[metadataDf['Trial'].isin(trial_window)].copy()
    keep_mask = np.array([not np.isnan(tsDataset[idx]).all() for idx in pool['tsIDx']])
    pool = pool[keep_mask]
    return (pool
            .groupby(['Age', 'Chrom', 'Parcel'])['Infant']
            .apply(set)
            .reset_index(name='raw_pool_infants'))


# --- Compute per-cell attrition ---
per_cell_rows = []
for test_name, trial_window in CONTRAST_TRIAL_WINDOWS.items():
    csv_path = RESULTS_PATHS.get(test_name)
    if csv_path is None or not os.path.exists(csv_path):
        continue

    results_df = pd.read_csv(csv_path)
    tested_by_cell = (results_df
                       .groupby(['Age', 'Chrom', 'Parcel'])['Infant']
                       .apply(set)
                       .reset_index(name='tested_infants'))

    raw_pool = _raw_pool_by_cell(trial_window)

    merged = raw_pool.merge(tested_by_cell,
                             on=['Age', 'Chrom', 'Parcel'], how='left')
    merged['tested_infants'] = merged['tested_infants'].apply(
        lambda x: x if isinstance(x, set) else set())
    merged['n_raw_pool']   = merged['raw_pool_infants'].apply(len)
    merged['n_tested']     = merged['tested_infants'].apply(len)
    merged['pct_tested_of_raw']  = 100.0 * merged['n_tested'] / merged['n_raw_pool']

    # For contrasts, also merge with activation-significant pool
    if test_name != 'activation':
        merged = merged.merge(act_sig_by_cell,
                               on=['Age', 'Chrom', 'Parcel'], how='left')
        merged['act_sig_infants'] = merged['act_sig_infants'].apply(
            lambda x: x if isinstance(x, set) else set())
        merged['tested_and_act_sig'] = merged.apply(
            lambda r: r['tested_infants'] & r['act_sig_infants'], axis=1)
        merged['n_act_sig']  = merged['act_sig_infants'].apply(len)
        merged['n_tested_of_act_sig'] = merged['tested_and_act_sig'].apply(len)
        merged['pct_tested_of_act_sig'] = 100.0 * merged['n_tested_of_act_sig'] / merged['n_act_sig'].replace(0, np.nan)
    else:
        merged['n_act_sig'] = np.nan
        merged['n_tested_of_act_sig'] = np.nan
        merged['pct_tested_of_act_sig'] = np.nan

    merged['test'] = test_name
    per_cell_rows.append(merged[['test', 'Age', 'Chrom', 'Parcel',
                                  'n_raw_pool', 'n_tested', 'pct_tested_of_raw',
                                  'n_act_sig', 'n_tested_of_act_sig', 'pct_tested_of_act_sig']])

perParcelDf = pd.concat(per_cell_rows, ignore_index=True)

# --- Summary across parcels: median (IQR) per (Age × Chrom × Test) ---
print("=" * 110)
print("Per-parcel attrition — medians across parcels within (Age × Chrom)")
print("=" * 110)
for test_name in CONTRAST_TRIAL_WINDOWS:
    sub = perParcelDf[perParcelDf['test'] == test_name]
    if sub.empty:
        continue
    print(f"\n--- {test_name} ---")
    grp = (sub.groupby(['Age', 'Chrom'])
             .agg(n_parcels=('Parcel', 'nunique'),
                  n_raw_pool_median=('n_raw_pool', 'median'),
                  n_tested_median=('n_tested', 'median'),
                  pct_tested_of_raw_median=('pct_tested_of_raw', 'median'),
                  n_act_sig_median=('n_act_sig', 'median'),
                  n_tested_of_act_sig_median=('n_tested_of_act_sig', 'median'),
                  pct_tested_of_act_sig_median=('pct_tested_of_act_sig', 'median'))
             .reset_index())
    print(grp.to_string(index=False))

perParcelDf.to_csv(DATA + "ATTRITION_per_parcel.csv", index=False)
print(f"\nWritten: {DATA}ATTRITION_per_parcel.csv")

In [ ]:
# =============================================================================
# Per-infant attrition — an infant counts as "not excluded" if their test could
# run at ANY parcel. Same dual denominator as per-parcel:
#   (i)  fraction of raw-pool infants who reached testing at ≥1 parcel
#   (ii) fraction of activation-significant infants who reached testing at ≥1 parcel
# =============================================================================

# --- Raw pool per (Age, Chrom): infants with any non-NaN trial at any parcel
#     within the contrast's trial window ---
def _raw_pool_infants(trial_window):
    pool = metadataDf[metadataDf['Trial'].isin(trial_window)].copy()
    keep_mask = np.array([not np.isnan(tsDataset[idx]).all() for idx in pool['tsIDx']])
    pool = pool[keep_mask]
    return (pool
            .groupby(['Age', 'Chrom'])['Infant']
            .apply(set)
            .reset_index(name='raw_pool_infants'))


# --- Activation-significant infants (any parcel) per (Age, Chrom) ---
act_sig_infants_by_ac = (act_sig
                          .groupby(['Age', 'Chrom'])['Infant']
                          .apply(set)
                          .reset_index(name='act_sig_infants'))


# --- Compute per-infant attrition ---
per_infant_rows = []
for test_name, trial_window in CONTRAST_TRIAL_WINDOWS.items():
    csv_path = RESULTS_PATHS.get(test_name)
    if csv_path is None or not os.path.exists(csv_path):
        continue

    results_df = pd.read_csv(csv_path)
    tested_infants_by_ac = (results_df
                             .groupby(['Age', 'Chrom'])['Infant']
                             .apply(set)
                             .reset_index(name='tested_infants'))

    raw_pool = _raw_pool_infants(trial_window)
    merged = raw_pool.merge(tested_infants_by_ac, on=['Age', 'Chrom'], how='left')
    merged['tested_infants'] = merged['tested_infants'].apply(
        lambda x: x if isinstance(x, set) else set())
    merged['n_raw_pool'] = merged['raw_pool_infants'].apply(len)
    merged['n_tested']   = merged['tested_infants'].apply(len)
    merged['pct_tested_of_raw']  = 100.0 * merged['n_tested'] / merged['n_raw_pool']

    # For contrasts, also compute vs activation-significant pool
    if test_name != 'activation':
        merged = merged.merge(act_sig_infants_by_ac, on=['Age', 'Chrom'], how='left')
        merged['act_sig_infants'] = merged['act_sig_infants'].apply(
            lambda x: x if isinstance(x, set) else set())
        merged['tested_and_act_sig'] = merged.apply(
            lambda r: r['tested_infants'] & r['act_sig_infants'], axis=1)
        merged['n_act_sig'] = merged['act_sig_infants'].apply(len)
        merged['n_tested_of_act_sig']   = merged['tested_and_act_sig'].apply(len)
        merged['pct_tested_of_act_sig'] = 100.0 * merged['n_tested_of_act_sig'] / merged['n_act_sig'].replace(0, np.nan)
    else:
        merged['n_act_sig'] = np.nan
        merged['n_tested_of_act_sig'] = np.nan
        merged['pct_tested_of_act_sig'] = np.nan

    merged['test'] = test_name
    per_infant_rows.append(merged[['test', 'Age', 'Chrom',
                                     'n_raw_pool', 'n_tested', 'pct_tested_of_raw',
                                     'n_act_sig', 'n_tested_of_act_sig', 'pct_tested_of_act_sig']])

perInfantDf = pd.concat(per_infant_rows, ignore_index=True)

print("=" * 110)
print("Per-infant attrition — one row per (Age × Chrom × Test)")
print("=" * 110)
for test_name in CONTRAST_TRIAL_WINDOWS:
    sub = perInfantDf[perInfantDf['test'] == test_name]
    if sub.empty:
        continue
    print(f"\n--- {test_name} ---")
    for _, row in sub.iterrows():
        line = (f"  {int(row['Age']):>3}mo Chrom={int(row['Chrom'])}: "
                f"raw_pool={int(row['n_raw_pool']):>3}, "
                f"tested={int(row['n_tested']):>3} "
                f"({row['pct_tested_of_raw']:.1f}% of raw)")
        if test_name != 'activation' and not np.isnan(row['n_act_sig']):
            line += (f"  |  act_sig_pool={int(row['n_act_sig']):>3}, "
                     f"tested_of_act_sig={int(row['n_tested_of_act_sig']):>3} "
                     f"({row['pct_tested_of_act_sig']:.1f}% of act-sig)")
        print(line)

perInfantDf.to_csv(DATA + "ATTRITION_per_infant.csv", index=False)
print(f"\nWritten: {DATA}ATTRITION_per_infant.csv")

In [ ]:
# =============================================================================
# GROUP-LEVEL TESTS
# =============================================================================
# Per (Parcel, Age, Chrom) cell with n_infants >= MIN_INFANTS:
#   a.  Direction-consistency binomial vs 0.5, one-sided.
#       "In direction" = signed TFCE peak aligns with the chromophore's
#       hypothesised sign (HbO +1, HbR -1).
#   b  Directional-hit-count vs empirical null.
#       n_hits  = # infants where per-infant sig flag is True.
#       p0      = mean of contributing infants' 'Null dir hit rate'
#                 (falls back to 0.025 if the column is unavailable).
#
# BH within each (Age, Chrom) stratum, at q = FDR_Q.
# =============================================================================

CHROM_DIRECTION = {1: +1, 2: -1}   # HbO positive, HbR negative
MIN_INFANTS     = 6
FDR_Q           = 0.05
SIGNED_STAT     = 'TFCE peak'          # signed per-infant statistic
NULL_RATE       = 'Null dir hit rate'  # per-infant empirical null hit-rate


def benjamini_hochberg_reject(pvals, q=FDR_Q):
    p = np.asarray(pvals, dtype=float)
    finite = ~np.isnan(p)
    reject = np.zeros(p.shape, dtype=bool)
    crit = 0.0
    pf = p[finite]
    m = pf.size
    if m == 0:
        return reject, crit
    order = np.argsort(pf)
    pf_sorted = pf[order]
    below = pf_sorted <= (np.arange(1, m + 1) / m) * q
    if below.any():
        k = int(np.max(np.where(below)[0]))
        crit = float(pf_sorted[k])
        full = np.zeros(m, dtype=bool)
        full[pf <= crit] = True
        reject[finite] = full
    return reject, crit


def group_level_direction_tests(df, sig_col, test_name):
    if df is None or df.empty:
        return pd.DataFrame()
    if SIGNED_STAT not in df.columns:
        raise ValueError(f"[{test_name}] missing column '{SIGNED_STAT}'")

    df = df.copy()
    df['_dir'] = df['Chrom'].map(CHROM_DIRECTION)
    if df['_dir'].isna().any():
        unknown = sorted(df.loc[df['_dir'].isna(), 'Chrom'].unique())
        print(f"  [{test_name}] chroms without CHROM_DIRECTION entry: {unknown} -- dropped")
        df = df[~df['_dir'].isna()].copy()
    if df.empty:
        return pd.DataFrame()

    # df['_in_direction'] = (df[SIGNED_STAT] * df['_dir']) > 0
    # df['_dir_hit']      = df[sig_col].astype(bool) & df['_in_direction']

    if test_name == 'activation':
        # activation's TFCE peak is already oriented upstream (chrom_sign
        # applied in maxstat_tfce_permutation): positive => hypothesised
        # direction for BOTH chromophores. Don't re-flip.
        df['_in_direction'] = df[SIGNED_STAT] > 0
    else:
        # contrasts store the raw signed effect; apply per-chrom direction.
        df['_in_direction'] = (df[SIGNED_STAT] * df['_dir']) > 0
    df['_dir_hit'] = df[sig_col].astype(bool) & df['_in_direction']

    rows = []
    for (age, chrom, parcel), cell in df.groupby(['Age', 'Chrom', 'Parcel']):
        usable = cell[~cell[SIGNED_STAT].isna()]
        n = len(usable)
        if n < MIN_INFANTS:
            continue

        # Direction-consistency test
        n_in = int(usable['_in_direction'].sum())
        p_sign = binomtest(n_in, n, 0.5, alternative='greater').pvalue

        # Directional-hit-count test
        n_hits = int(usable['_dir_hit'].sum())
        if NULL_RATE in usable.columns and usable[NULL_RATE].notna().any():
            p0 = float(np.nanmean(usable[NULL_RATE]))
        else:
            p0 = np.nan
        if np.isnan(p0) or p0 <= 0 or p0 >= 1:
            p0_use, p0_source = 0.025, 'fallback_0.025'
        else:
            p0_use, p0_source = p0, 'empirical'
        p_hit = binomtest(n_hits, n, p0_use, alternative='greater').pvalue

        rows.append({
            'test': test_name,
            'Age': age, 'Chrom': chrom, 'Parcel': parcel,
            'n_infants': n,
            'n_in_direction': n_in,
            'frac_in_direction': n_in / n,
            'p_direction_consistency': p_sign,
            'n_dir_hits': n_hits,
            'p0_used': p0_use,
            'p0_source': p0_source,
            'p_dir_hit_count': p_hit,
            'median_signed_TFCE_peak': float(np.nanmedian(usable[SIGNED_STAT])),
        })

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    out['sig_direction_consistency_fdr'] = False
    out['sig_directional_hit_count_fdr'] = False
    out['crit_p_direction_consistency']  = np.nan
    out['crit_p_directional_hit_count']  = np.nan
    for (age, chrom), idx in out.groupby(['Age', 'Chrom']).groups.items():
        block = out.loc[idx]
        rej_a, crit_a = benjamini_hochberg_reject(block['p_direction_consistency'].values)
        rej_b, crit_b = benjamini_hochberg_reject(block['p_dir_hit_count'].values)
        out.loc[idx, 'sig_direction_consistency_fdr'] = rej_a
        out.loc[idx, 'sig_directional_hit_count_fdr'] = rej_b
        out.loc[idx, 'crit_p_direction_consistency']  = crit_a
        out.loc[idx, 'crit_p_directional_hit_count']  = crit_b

    return out.sort_values(['Age', 'Chrom', 'p_direction_consistency']).reset_index(drop=True)


all_grouplv = []
for test_name, path, sig_col in TESTS:
    print(f"\n=== {test_name} (group-level) ===")
    df = _load(path)
    if df.empty or sig_col not in df.columns:
        continue
    gl = group_level_direction_tests(df, sig_col, test_name)
    if gl.empty:
        print(f"  No cells with n_infants >= {MIN_INFANTS} for {test_name}.")
        continue
    all_grouplv.append(gl)
    summary = gl.groupby(['Age', 'Chrom']).agg(
        cells=('Parcel', 'size'),
        sig_direction_consistency=('sig_direction_consistency_fdr', 'sum'),
        sig_directional_hit_count=('sig_directional_hit_count_fdr', 'sum'),
        median_n=('n_infants', 'median'),
    )
    print(f"  Group-level direction-consistency/directional-hit-count (cells with n>={MIN_INFANTS}):")
    print(summary.to_string())

groupLevelDf = pd.concat(all_grouplv, ignore_index=True) if all_grouplv else pd.DataFrame()
if not groupLevelDf.empty:
    groupLevelDf.to_csv(DATA + "GROUPLEVEL_direction_tests_all_tests.csv", index=False)
    print(f"\nWritten: {DATA}GROUPLEVEL_direction_tests_all_tests.csv")
else:
    print("\nNo group-level rows produced.")


In [ ]:
def _load_masked(path):
    if os.path.exists(path):
        return pd.read_csv(path)
    print(f"  Masked file not found: {path}")
    return pd.DataFrame()

habMaskedDf      = _load_masked(DATA + "surf_parcel_TFCE_habituation_MASKED_INDIV_spatiotemporal_perm_label.csv")
delayHabMaskedDf = _load_masked(DATA + "surf_parcel_TFCE_delayedHab_MASKED_INDIV_spatiotemporal_perm_label.csv")
lrHabMaskedDf    = _load_masked(DATA + "surf_parcel_TFCE_lrHab_MASKED_INDIV_spatiotemporal_perm_label.csv")

In [ ]:
all_union_hab = []
uniqueAges = metadataDf['Age'].unique()

for age in uniqueAges:
    union_frames = []
    for masked_df in (habMaskedDf, delayHabMaskedDf, lrHabMaskedDf):
        if isinstance(masked_df, pd.DataFrame) and not masked_df.empty and \
                'Masked significant' in masked_df.columns:
            age_sig = masked_df[(masked_df['Age'] == age) &
                                (masked_df['Masked significant'] == True)]
            if not age_sig.empty:
                union_frames.append(age_sig[['Infant', 'Parcel', 'Chrom', 'Age']])

    if union_frames:
        union_hab_df = pd.concat(union_frames).drop_duplicates()
        all_union_hab.append(union_hab_df.copy())

if all_union_hab:
    combined_union_hab = pd.concat(all_union_hab, ignore_index=True)
    combined_union_hab.to_csv(outputFilenameHabCombos, index=False)
    print(f"Written: {outputFilenameHabCombos} ({len(combined_union_hab)} rows)")
else:
    combined_union_hab = pd.DataFrame()
    print("No union habituation rows across ages — nothing written.")

In [ ]:
outputFilenameHabCombos

In [ ]:
all_union_hab

In [ ]:
combined_union_hab